## Гетерогенные эффекты: мета-обучатели, причинный лес и проверка оценки CATE

Практика к модулю [«Гетерогенные эффекты»](https://ai-interpretability.school/learn/5/0/0) школы AI Interpretability School, уроки «Мета-обучатели: S, T и X» и «Причинный лес и честное расщепление».

В модуле мы разобрали, почему индивидуальный эффект $\tau_i = Y_i(1) - Y_i(0)$ не наблюдается ни у кого, как условный средний эффект $\tau(x)$ оценивают мета-обучатели и причинный лес и почему проверить оценку можно только на группах объектов. В этой тетради все это проделывается на синтетических данных, где истинный $\tau(x)$ известен:

1. S-, T- и X-learner на EconML и рассеяние «истинный эффект против оцененного»;
2. причинный лес `CausalForestDML` с доверительными интервалами;
3. калибровка по квантилям предсказанного эффекта;
4. uplift-кривая, кривая Qini и коэффициент Qini;
5. DR-learner руками: дважды робастный псевдоисход с cross-fitting;
6. где мета-обучатели ломаются: S-learner с лассо и DR-learner при крайних склонностях.

На реальных данных сравнение с истинным эффектом невозможно, поэтому шаги 3 и 4 — те проверки, которые остаются в работе с настоящими данными, а шаги 1 и 2 показывают, насколько они согласуются с истиной.

In [ ]:
!pip install -q econml==0.17.0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso
from sklearn.model_selection import KFold
from econml.metalearners import SLearner, TLearner, XLearner
from econml.dml import CausalForestDML
from econml.dr import DRLearner

### Данные

Рандомизированный эксперимент: воздействие получает случайная пятая часть объектов, вероятность $0.2$ не зависит от признаков. Признаков пять, $X \sim \mathcal N(0, I)$:

- Эффект зависит только от первого признака: $\tau(x) = 1 + 1.5\,x_0$. У объектов с $x_0 < -2/3$ эффект отрицательный: воздействие им вредит.
- Исход без воздействия зависит от второго и третьего признаков: $\mu_0(x) = 2x_1 + x_2^2$. Эти признаки объясняют исход, но эффект не различают — как возраст в примере из урока «Причинный лес и честное расщепление».
- Наблюдаемый исход: $Y = \mu_0(X) + T\,\tau(X) + \varepsilon$, $\varepsilon \sim \mathcal N(0, 1)$.

Половина выборки идет на обучение, половина — на отложенную проверку.

In [ ]:
rng = np.random.default_rng(0)
n, d = 4000, 5
X = rng.normal(size=(n, d))
tau = 1 + 1.5 * X[:, 0]                       # истинный CATE
T = rng.binomial(1, 0.2, size=n)              # воздействие: случайная пятая часть
mu0 = 2 * X[:, 1] + X[:, 2] ** 2              # исход без воздействия
Y = mu0 + T * tau + rng.normal(size=n)

idx = rng.permutation(n)
tr, te = idx[: n // 2], idx[n // 2 :]
print('обучение:', len(tr), '|', 'проверка:', len(te), '|', 'доля воздействия:', round(T.mean(), 3))
print('доля объектов с отрицательным эффектом:', round((tau < 0).mean(), 3))

**Задание 1. Обучите S-, T- и X-learner из `econml.metalearners` на обучающей половине. Внутри всех схем — `GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=0)`, у X-learner модель склонности — `LogisticRegression()`. Посчитайте оценки CATE на проверочной половине (`tau_s`, `tau_t`, `tau_x`) и RMSE каждой относительно истинного `tau[te]`. Какая схема ближе всего к истинному эффекту?**

Оформите RMSE функцией `rmse(est)` и модель внутри схем — функцией `gb()`: они понадобятся дальше.

In [ ]:
# Ваш код здесь

Рассеяние «истинный эффект против оцененного»: по оси X истинный $\tau(x)$, по оси Y оценка схемы, красная линия — $y = x$. Чем ближе облако точек к линии, тем точнее схема восстанавливает эффект на отдельных объектах. Такую картинку можно нарисовать только на синтетических данных.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharex=True, sharey=True)
for ax, (name, est) in zip(axes, (('S-learner', tau_s), ('T-learner', tau_t), ('X-learner', tau_x))):
    ax.scatter(tau[te], est, s=6, alpha=0.35, color='#3b6ea5')
    lim = [tau[te].min(), tau[te].max()]
    ax.plot(lim, lim, color='#c0392b', lw=1.5, label='y = x')
    ax.set_title(f'{name}: RMSE = {rmse(est):.2f}')
    ax.set_xlabel('истинный эффект τ(x)')
    ax.legend(loc='upper left')
axes[0].set_ylabel('оценка эффекта τ̂(x)')
fig.suptitle('Истинный и оцененный CATE на отложенной половине')
fig.tight_layout()
fig.savefig('cate_true_vs_estimated.png', dpi=150, bbox_inches='tight')
plt.show()

**Задание 2. Обучите причинный лес `CausalForestDML` на обучающей половине: `model_y=gb()`, `model_t=GradientBoostingClassifier(random_state=0)`, `discrete_treatment=True`, `n_estimators=1000`, `min_samples_leaf=10`, `random_state=0`; честное расщепление в нем включено по умолчанию (`honest=True`). Посчитайте на проверочной половине оценку `tau_cf`, ее RMSE и 95-процентные доверительные интервалы `effect_interval(..., alpha=0.05)`. Какая доля объектов проверочной половины имеет истинный эффект внутри своего интервала?**

In [ ]:
# Ваш код здесь

### Калибровка по квантилям предсказанного эффекта

Дальше — проверки, которые не требуют знать истинный эффект. Отсортируем объекты проверочной половины по предсказанному эффекту и разобьем на четыре группы по четверти. Внутри каждой группы посчитаем фактическую разность средних исходов между воздействованными и контрольными: эксперимент рандомизирован, поэтому эта разность оценивает средний эффект в группе. Если модель ранжирует эффект осмысленно, фактическая разность растет вместе с предсказанной.

**Задание 3. Разбейте проверочную половину на квартили по `tau_cf` (`pd.qcut(tau_cf, 4, labels=[1, 2, 3, 4])`). Для каждого квартиля выведите средний предсказанный эффект, фактическую разность средних исходов воздействованных и контрольных и, для сравнения, средний истинный эффект. Растет ли фактическая разность от первого квартиля к четвертому?**

In [ ]:
# Ваш код здесь

### Uplift-кривая и кривая Qini

Калибровка по квартилям — четыре точки. Uplift-кривая делает то же для любой доли объектов: сортируем объекты по $\hat\tau(x)$ по убыванию, берем верхнюю долю $\varphi$ и считаем, какой прирост исхода на один объект всей выборки дало бы воздействие только на эту долю: $(\bar Y_{T=1}(\varphi) - \bar Y_{T=0}(\varphi))\cdot\varphi$. Случайный порядок дает прямую от нуля до разности средних исходов воздействованных и контрольных по всей выборке.

Кривая Qini устроена так же, но считает прирост через суммы: $Q(\varphi) = \bigl(S_{T=1}(\varphi) - S_{T=0}(\varphi)\cdot N_{T=1}(\varphi)/N_{T=0}(\varphi)\bigr)/N_{T=1}$, где $S$ — сумма исходов, $N$ — число объектов группы среди верхней доли $\varphi$. **Коэффициент Qini** — площадь между кривой Qini и прямой случайного порядка: чем она больше, тем лучше модель отделяет объекты с большим эффектом от объектов с малым.

**Задание 4. Напишите функцию `qini_curve(tau_hat, y, t)`, которая возвращает долю $\varphi$ (от 0 до 1 с шагом в один объект), значения $Q(\varphi)$ и коэффициент Qini — площадь $\int_0^1 \bigl(Q(\varphi) - \varphi\,Q(1)\bigr)\,d\varphi$ по формуле трапеций (`np.trapezoid`). Посчитайте коэффициент на проверочной половине для причинного леса, для X-learner и для сортировки по истинному эффекту `tau[te]`.**

In [ ]:
# Ваш код здесь

Слева — uplift-кривые причинного леса, X-learner и сортировки по истинному эффекту, пунктир — случайный порядок. Справа — кривая Qini причинного леса; закрашенная площадь между ней и прямой случайного порядка и есть коэффициент Qini.

In [ ]:
def uplift_curve(tau_hat, y, t):
    order = np.argsort(-tau_hat)
    y, t = y[order], t[order]
    n_t, n_c = np.cumsum(t), np.cumsum(1 - t)
    mean_t = np.divide(np.cumsum(y * t), n_t, out=np.zeros(len(y)), where=n_t > 0)
    mean_c = np.divide(np.cumsum(y * (1 - t)), n_c, out=np.zeros(len(y)), where=n_c > 0)
    k = np.arange(1, len(y) + 1)
    gain = np.where((n_t > 0) & (n_c > 0), (mean_t - mean_c) * k / len(y), 0.0)
    return np.r_[0, k / len(y)], np.r_[0, gain]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6))
for name, est, color in (('причинный лес', tau_cf, '#3b6ea5'), ('X-learner', tau_x, '#e67e22'),
                         ('Sorted by true effect', tau[te], '#7f8c8d')):
    phi, g = uplift_curve(est, y_te, t_te)
    ax1.plot(phi, g, color=color, lw=1.8, ls='--' if name.startswith('Sorted') else '-', label=name)
ax1.plot([0, 1], [0, g[-1]], color='black', lw=1, ls=':', label='случайный порядок')
ax1.set_title('uplift-кривая')
ax1.set_xlabel('Доля объектов, получивших воздействие, по убыванию предсказанного эффекта')
ax1.set_ylabel('Накопленный выигрыш на объект')
ax1.legend()

phi, q, coef = qini_curve(tau_cf, y_te, t_te)
ax2.plot(phi, q, color='#3b6ea5', lw=1.8, label='причинный лес')
ax2.plot([0, 1], [0, q[-1]], color='black', lw=1, ls=':', label='случайный порядок')
ax2.fill_between(phi, phi * q[-1], q, color='#3b6ea5', alpha=0.2, label=f'коэффициент Qini = {coef:.3f}')
ax2.set_title('кривая Qini')
ax2.set_xlabel('Доля объектов, получивших воздействие, по убыванию предсказанного эффекта')
ax2.set_ylabel('Qini: прирост выигрыша на объект с воздействием')
ax2.legend()
fig.tight_layout()
fig.savefig('uplift_qini_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### DR-learner: дважды робастный псевдоисход

X-learner достраивает недостающий потенциальный исход моделью и смешивает две модели эффекта по склонности. DR-learner ([Kennedy, 2023](https://arxiv.org/abs/2004.14497)) строит для каждого объекта одно число — псевдоисход, у которого условное среднее при верных вспомогательных моделях равно эффекту, — и регрессирует его на признаки:

$$\varphi_i = \hat\mu_1(X_i) - \hat\mu_0(X_i) + \frac{T_i\,(Y_i - \hat\mu_1(X_i))}{\hat e(X_i)} - \frac{(1-T_i)\,(Y_i - \hat\mu_0(X_i))}{1-\hat e(X_i)}.$$

Здесь $\hat\mu_1$ и $\hat\mu_0$ — модели исхода, обученные на воздействованных и на контрольных, $\hat e$ — модель склонности. Псевдоисход объекта считают моделями, которые этого объекта не видели (cross-fitting): выборку делят на пять фолдов, и для объектов каждого фолда вспомогательные модели обучают на остальных четырех.

Эксперимент в тетради рандомизирован, и настоящая склонность равна $0.2$ у всех; модель склонности все равно обучаем, как пришлось бы делать на наблюдательных данных.

**Задание 5. Напишите функцию `dr_pseudo(Xa, Ta, Ya, prop_model, clip=None)`, которая возвращает псевдоисход $\varphi_i$ и склонность $\hat e(X_i)$ для каждого объекта выборки `(Xa, Ta, Ya)`. Разбиение — `KFold(5, shuffle=True, random_state=0)`; в каждом фолде модели исхода `gb()` обучаются отдельно на воздействованных и на контрольных объектах остальных фолдов, модель склонности — `prop_model()` на всех объектах остальных фолдов; если задан `clip`, склонность обрезается до отрезка $[\text{clip}, 1-\text{clip}]$. Посчитайте псевдоисход на обучающей половине с `LogisticRegression` и обучите на нем две модели эффекта: `LinearRegression()` и `gb()`. Выведите RMSE обеих оценок на проверочной половине (`tau_dr_lin`, `tau_dr_gb`) и, для сравнения, RMSE X-learner и `econml.dr.DRLearner` с `model_regression=gb()`, `model_propensity=LogisticRegression()`, `model_final=gb()`, `cv=5`, `random_state=0`.**

In [ ]:
# Ваш код здесь

Псевдоисход шумный: у воздействованного объекта остаток исхода делится на склонность около $0.2$, то есть умножается примерно на пять. Линейная модель эффекта такой шум усредняет и к тому же совпадает по классу с истинным $\tau(x)=1+1.5\,x_0$, поэтому ее ошибка мала. Бустинг глубины 3 подгоняется под шум псевдоисхода и ошибается сильнее X-learner — тот же вывод, что в тетради «R-learner»: модели эффекта нужна простая форма или сильная регуляризация.

`econml.dr.DRLearner` устроен не так, как схема Kennedy: вместо двух моделей исхода по группам он обучает одну модель `model_regression` на признаках и воздействии вместе (как S-learner) и обрезает склонность снизу и сверху на `min_propensity=1e-6`. Поэтому число библиотеки не совпадает с нашим.

### Где ломается: S-learner с лассо

Урок «Мета-обучатели: S, T и X» предупреждает: для S-learner воздействие — один признак из многих, и регуляризация может свести его коэффициент к нулю. Проверим на линейном S-learner: лассо на признаках $[X, T]$. В модели без взаимодействий оценка эффекта одна на всех — коэффициент при $T$.

**Задание 6. Для каждого `alpha` из `(0.01, 0.03, 0.1, 0.2, 0.3, 0.5, 1.0)` обучите `Lasso(alpha=alpha)` на признаках `np.c_[X[tr], T[tr]]` и исходе `Y[tr]` и выведите таблицу: коэффициент при $T$ (оценка эффекта S-learner), коэффициент при $x_1$ и число ненулевых коэффициентов при признаках $x_0,\dots,x_4$. При каком наименьшем `alpha` из перебора оценка эффекта становится ровно нулевой и что в этот момент с коэффициентом при $x_1$?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

### Где ломается: DR-learner при крайних склонностях

Слабое место DR-learner — то же, что у IPW: деление на $\hat e(x)$ и $1-\hat e(x)$. Возьмем наблюдательные данные, где воздействие зависит от того же признака, что и эффект: склонность $e(x)=1/(1+e^{-3x_0})$, у объектов с большим $|x_0|$ она близка к 0 или 1. Эффект и исход без воздействия те же, что выше; 2000 объектов на обучение, 2000 на проверку.

In [ ]:
rc = np.random.default_rng(1)
Xc = rc.normal(size=(n, d))
tau_c = 1 + 1.5 * Xc[:, 0]
Tc = rc.binomial(1, 1 / (1 + np.exp(-3 * Xc[:, 0])))       # склонность растет с x0
Yc = 2 * Xc[:, 1] + Xc[:, 2] ** 2 + Tc * tau_c + rc.normal(size=n)
tr_c, te_c = np.arange(n // 2), np.arange(n // 2, n)
rmse_c = lambda est: np.sqrt(np.mean((est - tau_c[te_c]) ** 2))
print('доля воздействия:', round(Tc.mean(), 3))

**Задание 7. На обучающей части `tr_c` посчитайте псевдоисход функцией `dr_pseudo` с `LogisticRegression` дважды: без обрезки и с `clip=0.05`. Для каждого варианта выведите долю объектов, у которых склонность до обрезки лежит вне отрезка $[0.05, 0.95]$, наибольший $|\varphi_i|$ и RMSE DR-learner с моделью эффекта `gb()` на `te_c`. Для сравнения выведите RMSE X-learner с `gb()` и `LogisticRegression()`. Что делает обрезка и какой ценой?**

In [ ]:
# Ваш код здесь

**Ваш ответ:**

**Выводы:**

- На синтетике видно то, что в работе с реальными данными недоступно: насколько каждая оценка $\hat\tau(x)$ отличается от истинного эффекта на отдельном объекте. Сравните RMSE схем с тем, что урок «Мета-обучатели: S, T и X» говорит про дисбаланс групп: воздействие здесь получила пятая часть выборки.
- Доверительные интервалы причинного леса — асимптотические. Сравните долю объектов, у которых истинный эффект попал в интервал, с номинальными 95%.
- Калибровка по квантилям, uplift-кривая и коэффициент Qini не требуют знать истинный эффект, но проверяют только ранжирование по группам, а не точность на объекте.
- Признаки $x_1$ и $x_2$ определяют исход, но не эффект: модель исхода считала бы их важными, а для решения «кому назначать воздействие» они ничего не дают.
- DR-learner сводит оценку CATE к регрессии псевдоисхода, но псевдоисход шумный: модели эффекта нужна простая форма, а склонность — вдали от 0 и 1.
- S-learner с лассо может обнулить эффект целиком, сохранив признаки, которые лучше объясняют исход.

Источники:
1. [Künzel et al., 2019. Metalearners for estimating heterogeneous treatment effects using machine learning](https://arxiv.org/abs/1706.03461).
2. [Wager и Athey, 2018. Estimation and inference of heterogeneous treatment effects using random forests](https://arxiv.org/abs/1510.04342).
3. [Kennedy, 2023. Towards optimal doubly robust estimation of heterogeneous causal effects](https://arxiv.org/abs/2004.14497).
4. [EconML](https://github.com/py-why/EconML).